<h1 style="color: #9f43c3ff; border-bottom: 3px solid #9f43c3ff; padding-bottom: 8px;">
AI & ML Course with BinX — Week 4 — Day 3: Bias-Variance Trade-Off & Regularization
</h1>

<blockquote style="border-left: 3px solid #2e1a9aff; padding-left: 12px; margin-left: 0;">

<b>Day 3 Learning Objectives:</b>
- <b>Diagnose High Variance:</b> Train an unconstrained Decision Tree and quantify the train-vs-test generalization gap.
- <b>Diagnose High Bias:</b> Restrict model capacity to a single decision stump and observe performance suppression.
- <b>Apply Pre-Pruning Regularization:</b> Calibrate hyperparameters (<code>max_depth</code>, <code>min_samples_leaf</code>) to reduce complexity.
- <b>Shrink the Generalization Gap:</b> Validate that regularizing the model reduces variance while improving test set generalization.

</blockquote>


## <span style="color: #309c42ff">3.1 Dataset Loading & Preprocessing</span>

<blockquote style="border-left: 3px solid #FD1D1D; padding-left: 12px; margin-left: 0;">

We load the Telco Customer Churn dataset (<code>Customer-Churn.csv</code>), coerce <code>TotalCharges</code> whitespace strings into numeric floats, drop null entries, remove the non-predictive <code>customerID</code>, and apply one-hot dummy encoding. Finally, we partition the dataset into a <b>70% Training / 30% Test</b> split with target stratification (<code>stratify=y</code>).

</blockquote>


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score

df = pd.read_csv('../../Data/Customer-Churn.csv')

In [2]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna()

x = df.drop(columns=['customerID', 'Churn'])
y = df['Churn']

x = pd.get_dummies(x, drop_first=True)

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.3, random_state=42, stratify=y
)

## <span style="color: #309c42ff">3.2 Step 1: Diagnosing Overfitting (High Variance)</span>

<blockquote style="border-left: 3px solid #FD1D1D; padding-left: 12px; margin-left: 0;">

<b>Hands-On Lab — Step 1:</b>
We train an unconstrained Decision Tree (<code>max_depth=50</code>). The tree grows deep branches that memorize noise and sample-specific patterns in the training data, leading to near-perfect training accuracy but a severe drop in test performance (large generalization gap).

</blockquote>


In [3]:
overfit_tree = DecisionTreeClassifier(max_depth=100, random_state=42)
overfit_tree.fit(x_train, y_train)

train_acc = accuracy_score(y_train, overfit_tree.predict(x_train))
test_acc = accuracy_score(y_test, overfit_tree.predict(x_test))

generalization_gap = train_acc - test_acc

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy:  {test_acc:.4f}")
print(f"Generalization Gap:  {generalization_gap:.4f}")

Train Accuracy: 0.9988
Test Accuracy:  0.7047
Generalization Gap:  0.2940


## <span style="color: #309c42ff">3.3 Step 2: Diagnosing Underfitting (High Bias)</span>

<blockquote style="border-left: 3px solid #FD1D1D; padding-left: 12px; margin-left: 0;">

<b>Hands-On Lab — Step 2:</b>
We restrict the model capacity to a single decision stump with <code>max_depth=1</code>. The model makes overly simplistic assumptions and lacks the structural depth to capture non-linear feature interactions, resulting in equally suppressed training and testing accuracy.

</blockquote>


In [6]:

underfit_tree = DecisionTreeClassifier(max_depth=1, random_state=42)
underfit_tree.fit(x_train, y_train)

train_acc = underfit_tree.score(x_train, y_train)
test_acc = underfit_tree.score(x_test, y_test)

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy:  {test_acc:.4f}")


Train Accuracy: 0.7343
Test Accuracy:  0.7341


## <span style="color: #309c42ff">3.4 Step 3: Regularization & Shrinking the Generalization Gap</span>

<blockquote style="border-left: 3px solid #FD1D1D; padding-left: 12px; margin-left: 0;">

<b>Hands-On Lab — Step 3:</b>
We apply pre-pruning regularization by constraining tree depth (<code>max_depth=3</code>) and enforcing a minimum leaf sample threshold (<code>min_samples_leaf=10</code>). This balances the bias-variance trade-off, shrinking the generalization gap while boosting unseen test performance.

</blockquote>


In [5]:
regularized_tree = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=10,
    random_state=42
)

regularized_tree.fit(x_train, y_train)
train_acc = regularized_tree.score(x_train, y_train)
test_acc = regularized_tree.score(x_test, y_test)

generalization_gap = train_acc - test_acc

print(f"Train Accuracy:      {train_acc:.4f}")
print(f"Test Accuracy:       {test_acc:.4f}")
print(f"Generalization Gap:  {generalization_gap:.4f}")

Train Accuracy:      0.7877
Test Accuracy:       0.7791
Generalization Gap:  0.0085


## <span style="color: #309c42ff">3.5 Experimental Comparison & Diagnostic Insights</span>

<blockquote style="border-left: 3px solid #FD1D1D; padding-left: 12px; margin-left: 0;">

<b>Summary of Findings:</b>
We benchmark the three models side-by-side to observe how controlling complexity resolves high variance without introducing harmful bias.

</blockquote>


| Model Architecture | Hyperparameters | Train Accuracy | Test Accuracy | Generalization Gap | Diagnostic State |
| :--- | :--- | :---: | :---: | :---: | :--- |
| **Unconstrained Tree** | `max_depth=50` | `0.9988` | `0.7047` | `0.2941 (29.41%)` | High Variance (Overfitting) |
| **Decision Stump** | `max_depth=1` | `0.7343` | `0.7341` | `0.0002 (0.02%)` | High Bias (Underfitting) |
| **Regularized Tree** | `max_depth=3, min_samples_leaf=10` | **`0.7877`** | **`0.7791`** | **`0.0085 (0.85%)`** | **Optimal Equilibrium** |

### Key Takeaway
Restricting tree depth and node sample limits reduced the generalization gap from **29.41% down to 0.85%**, while raising real-world test accuracy from **70.47% to 77.91%**.
